# 02 · Threads and waiting for I/O

**Time:** 25–30 minutes, including the exercises.
**Before you start:** Know functions, lists, loops, and the idea of I/O from lesson 01.

You will start threads, use a thread pool, collect results and errors, and protect shared data.
The examples use Python's standard library. Run the cells from top to bottom in a Python 3.10+ Jupyter kernel.

I/O means reading or writing outside your program, such as reading a file or asking a server for data.
These operations often involve waiting. We will mostly **simulate that waiting with `time.sleep`**.
No network connection or dataset is needed. These are learning examples, not performance benchmarks.

[Course guide](README.md) · Previous: [Concurrency and the GIL](01_concurrency_and_the_gil.ipynb) ·
Next: [Processes and CPU work](03_processes_and_cpu_work.ipynb)

In [1]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tempfile import TemporaryDirectory

jobs = ["image_0", "image_1", "image_2", "image_3"]

def pretend_read(name, delay=0.15):
    # Simulate waiting for data; no image is actually read or decoded.
    time.sleep(delay)
    return f"data for {name}"

## 1. Start with one job at a time

**Predict:** Four jobs each wait for 0.15 seconds. About how much waiting does this loop do?

`perf_counter()` measures elapsed time. The difference between two readings tells us how long a section took.

In [2]:
started = time.perf_counter()
sequential_results = [pretend_read(name) for name in jobs]
sequential_seconds = time.perf_counter() - started

print(sequential_results)
print(f"Sequential: {sequential_seconds:.3f} seconds")

['data for image_0', 'data for image_1', 'data for image_2', 'data for image_3']
Sequential: 0.603 seconds


**Explain:** The waits add up to about 0.60 seconds, plus overhead.
Nothing starts the next job while the current call sleeps.

## 2. Start threads, then join them

A thread is a separate path through your program. Threads in one process share its memory.
`start()` begins a thread; `join()` waits for it to finish.
Start all the threads before joining them so their work can overlap.

In `Thread(target=read_into_slot, args=(index, name))`, `target` is the function
to run and `args` supplies its arguments. We pass the function itself, without
calling it. `enumerate(jobs)` supplies each item's position and name.
[Python's thread documentation](https://docs.python.org/3/library/threading.html#thread-objects)

**Predict:** Will the four waits add up again, or overlap?

In [3]:
thread_results = [None] * len(jobs)

def read_into_slot(index, name):
    # Each worker owns a different slot in this shared list.
    thread_results[index] = pretend_read(name)

threads = [
    threading.Thread(target=read_into_slot, args=(index, name))
    for index, name in enumerate(jobs)
]

started = time.perf_counter()
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
threaded_seconds = time.perf_counter() - started

print(thread_results)
print(f"Threads: {threaded_seconds:.3f} seconds")
assert thread_results == sequential_results

['data for image_0', 'data for image_1', 'data for image_2', 'data for image_3']
Threads: 0.151 seconds


**Explain:** One thread can progress while another waits. Expect elapsed time closer to one wait,
but exact timings depend on your machine. Our assertion checks the returned data, not speed.

If we put `start()` and `join()` next to each other inside one loop, we would wait for each job
before starting the next one. Try that as an experiment, then restore the cell.

A `Thread` does not give you its function's return value. We used separate list slots to collect results.
`join()` also does not raise a worker's exception in the caller; an unhandled
worker error is printed separately. A thread pool makes results and errors easier to collect.

## 3. Let a pool manage the threads

A **worker** runs assigned jobs. A **pool** creates up to a set number of
workers and reuses them. `max_workers=2` allows up to two jobs to run at once.
`map(function, items)` applies the function to each item and produces results in
input order. `list(...)` collects them, waiting for results that are still pending.

The `with` block manages setup and cleanup; this is called a **context manager**.
Leaving this pool's block waits for its jobs to finish, including when an error occurs.
[Executor documentation](https://docs.python.org/3/library/concurrent.futures.html#executor-objects)

A **wave** is a group of jobs whose waits overlap.

**Predict:** With four equally slow jobs and two workers, about how many waves
of waiting will we need?

In [4]:
started = time.perf_counter()
with ThreadPoolExecutor(max_workers=2) as pool:
    pool_results = list(pool.map(pretend_read, jobs))
pool_seconds = time.perf_counter() - started

print(pool_results)
print(f"Two workers: {pool_seconds:.3f} seconds")
assert pool_results == sequential_results

['data for image_0', 'data for image_1', 'data for image_2', 'data for image_3']
Two workers: 0.302 seconds


**Explain:** We expect about two waves, or 0.30 seconds of waiting plus overhead.
Returning results in input order does not mean the jobs ran one at a time.

More workers are not always better. Servers may limit requests, disks have finite bandwidth,
and each thread has a cost. Measure with the real workload before choosing a worker count.

`max_workers` limits running workers, not how many pending jobs you can submit.
Avoid building millions of pending jobs at once. We will handle that problem in the backpressure lesson.

## 4. Handle each result when it is ready

`submit()` returns a **future**: a handle for a job's eventual result.
`as_completed()` produces finished futures; `result()` waits if necessary,
then returns a value or raises the worker's exception. An **exception** is
Python's way of reporting a failure; `try`/`except` lets the caller handle it.
If several jobs are already finished, do not depend on the exact order they are returned.
[Future documentation](https://docs.python.org/3/library/concurrent.futures.html#future-objects)

**Predict:** Can we still collect successful results if one input is invalid?

In [5]:
requests = [("slow", 0.18), ("fast", 0.03), ("broken", 0.06)]

def read_checked(name, delay):
    data = pretend_read(name, delay)
    if name == "broken":
        raise ValueError("This example input is intentionally invalid")
    return data

successful = {}
failed = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    future_to_name = {}
    for name, delay in requests:
        future = pool.submit(read_checked, name, delay)
        future_to_name[future] = name  # Remember which input belongs to this job.
    for future in as_completed(future_to_name):
        name = future_to_name[future]
        try:
            successful[name] = future.result()
            print("Finished:", name)
        except ValueError as error:
            failed[name] = str(error)
            print("Failed:", name, "-", error)

assert set(successful) == {"slow", "fast"}
assert set(failed) == {"broken"}

Finished: fast
Failed: broken - This example input is intentionally invalid
Finished: slow


**Explain:** Each future keeps its own outcome. The caller can decide what to do about a failure.
We catch the specific error used in this example, so unexpected errors stay visible.
`ValueError` reports an invalid value; `raise ValueError(...)` deliberately creates that failure.
With `map()`, a worker error is raised when you retrieve that item's result.

Call `result()` or otherwise inspect submitted futures. A worker exception can go unnoticed if you discard its future.

## 5. A tiny example with real files

These are ordinary file reads, not simulated waits. A temporary directory holds four small text files
and is removed when the block finishes. Each task opens its own file.

This shows the pattern. Files this small will often be faster to read sequentially because starting
and coordinating workers costs time, and the operating system may already have the data in memory.

In [6]:
def read_text(path):
    return path.read_text(encoding="utf-8")

with TemporaryDirectory() as directory:
    paths = [Path(directory) / f"sample_{i}.txt" for i in range(4)]
    for index, path in enumerate(paths):
        path.write_text(f"sample {index}: label=cat", encoding="utf-8")

    with ThreadPoolExecutor(max_workers=2) as pool:
        file_contents = list(pool.map(read_text, paths))

print(file_contents)

['sample 0: label=cat', 'sample 1: label=cat', 'sample 2: label=cat', 'sample 3: label=cat']


## 6. Shared memory can cause lost updates

A **race condition** happens when correctness depends on how operations from different workers overlap.
For example, two threads can read the same old counter value, each add one, and overwrite one another's update.

To make the mistake visible every time, we deliberately pause both workers after reading the counter.
`Barrier(2)` is a meeting point: each worker waits until both have arrived.
This extra coordination is only here to stage the mistake. It is not needed in ordinary counter code.

**Predict:** Two workers each add one. Will the final value be 1 or 2?

In [7]:
counter = {"completed": 0}
both_have_read = threading.Barrier(2)

def unsafe_increment():
    old_value = counter["completed"]
    both_have_read.wait(timeout=10)
    counter["completed"] = old_value + 1

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(unsafe_increment) for _ in range(2)]
    for future in futures:
        future.result()

print("Expected after two increments: 2")
print("Actual:", counter["completed"])
assert counter["completed"] == 1

Expected after two increments: 2
Actual: 1


**Explain:** Both workers read 0. Both then store 1. One update disappears.

The GIL does not turn a sequence of reads and writes into one indivisible operation.
A **lock** lets one worker at a time enter a protected section. All workers touching the shared
counter must follow the same locking rule. `with counter_lock:` acquires the
lock on entry and releases it on exit, including if an exception occurs.
[Lock documentation](https://docs.python.org/3/library/threading.html#lock-objects)

We remove the teaching barrier in the fix. Waiting for another worker at a barrier while holding
the lock it needs would stop both workers from making progress.

In [8]:
counter = {"completed": 0}
counter_lock = threading.Lock()

def safe_increment():
    with counter_lock:
        old_value = counter["completed"]
        counter["completed"] = old_value + 1

with ThreadPoolExecutor(max_workers=2) as pool:
    futures = [pool.submit(safe_increment) for _ in range(2)]
    for future in futures:
        future.result()

print("With a lock:", counter["completed"])
assert counter["completed"] == 2

With a lock: 2


Keep protected sections short. For independent work, an even simpler design is to return each
result to the caller and combine results there. That is how our earlier `pool.map()` example worked.

## Exercises

1. Run `pretend_read` on six names using one, two, and three workers. Predict the number of waves first.
   Confirm that all three settings return the same data. Are the measured times close to your estimate?
2. A teammate submits a job, calls `future.result()`, then submits the next job, repeating in a loop.
   Explain why that pattern loses concurrency. Rewrite it for a small list of jobs.
3. You need to count successful reads. Can you do that without a shared worker counter or a lock?

In [9]:
# Your work. Reuse pretend_read, ThreadPoolExecutor, and time.perf_counter.
exercise_jobs = [f"sample_{index}" for index in range(6)]

<details>
<summary>Worked answers — open after trying</summary>

**1.** Six jobs need roughly six, three, and two waves with one, two, and three workers.
The waiting parts are approximately 0.90, 0.45, and 0.30 seconds. These are estimates, not guarantees.

```python
expected = [f"data for {name}" for name in exercise_jobs]
for workers in (1, 2, 3):
    started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        results = list(pool.map(pretend_read, exercise_jobs))
    print(workers, round(time.perf_counter() - started, 3))
    assert results == expected
```

**2.** Calling `result()` immediately waits before the caller submits the next job.
Submit this small set first, then collect the results:

```python
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(pretend_read, name) for name in exercise_jobs]
    results = [future.result() for future in futures]
```

**3.** In the earlier error-handling example, `len(successful)` is the count.
Only the caller updates that dictionary. Workers return data through their futures.

</details>

## What to remember

- Threads help overlap waiting. On a typical CPython build with the GIL enabled, pure Python CPU
  loops usually do not become faster just because you add threads. Some native libraries release the GIL;
  optional free-threaded CPython builds are a separate case.
- Start threads before joining them. Prefer a pool for collections of independent jobs.
- `map()` preserves input order; `as_completed()` lets you handle finished futures as they become available.
- Inspect results so worker errors are visible. Protect shared updates or return data to a single collector.

**Machine learning engineer (MLE) connection:** Threads can overlap downloads or reads before preprocessing. Image decoding may
involve native code, so measure where its time goes instead of assuming every
decoder has the same GIL behavior.

Next: [03 · Processes and CPU work](03_processes_and_cpu_work.ipynb).

**Official references:** [Threads and the GIL](https://docs.python.org/3/library/threading.html),
[ThreadPoolExecutor and futures](https://docs.python.org/3/library/concurrent.futures.html).